# 1. Two Sum
**Difficulty:** 🟢 Easy · **Topic:** Array · **LeetCode:** https://leetcode.com/problems/two-sum/

## 💡 Concepts

**Core concept(s):** Hashing (hash map / dictionary) and the idea of a *complement*.

**Why they apply here:** For each number `x`, the pair that solves the problem is exactly `target - x` (its *complement*). Finding whether the complement already exists is a **membership test**. A hash map answers membership and returns the stored index in **O(1)** average time, so we can replace an inner scan with a single lookup.

**Key intuition / mental model:** Instead of asking “for this pair, do they sum to target?” (which compares every pair), flip it to “I have `x`; have I *already seen* the number that completes it?” As we walk the array once, we remember everything we've passed. The moment a number's complement is in our memory, we're done.

---

### 📚 What is a Hash Map?
A **hash map** (Python `dict`) stores **key → value** pairs. Internally it runs each key through a *hash function* that turns the key into an array index (a "bucket"), so it can jump **straight** to where the value lives instead of scanning every entry. When two keys land in the same bucket (a *collision*), it stores them together and does a tiny local search.
- **Key operations & complexity:** insert, lookup, and delete are all **O(1) on average** (degrading to O(n) only in the rare worst case of many collisions).
- **In Python:** use `dict` for key→value mapping, or `set` when you only care about membership (does this value exist?).
- **Why it matters here:** the O(1) lookup is exactly what lets us check "have I seen the complement?" inside a single loop — turning an O(n²) double scan into one O(n) pass.

### 📚 What is the Two-Pointer Technique?
**Two pointers** means keeping two indices that move through the data (often one from each end, or one fast + one slow) so you cover the array in a **single sweep** instead of nested loops. On a **sorted** array it works because order tells you which pointer to move: if the current sum is too small, move the left pointer right to grow it; if too big, move the right pointer left to shrink it.
- **Key operation & complexity:** one linear **O(n)** sweep, using **O(1)** extra space.
- **In Python:** just two integer index variables (e.g. `lo`, `hi`) — no special type needed.

---

**Prerequisite knowledge:**
- How a Python `dict`/`set` gives average O(1) insert and lookup (see primer above).
- The two-pointer technique on a *sorted* array (see primer above).

## 📝 Problem

Given an array of integers `nums` and an integer `target`, return the **indices** of the two numbers that add up to `target`. Each input has *exactly one* solution, and you may not use the same element twice.

**Example 1**
```
Input:  nums = [2, 7, 11, 15], target = 9
Output: [0, 1]        # nums[0] + nums[1] = 2 + 7 = 9
```
**Example 2**
```
Input:  nums = [3, 2, 4], target = 6
Output: [1, 2]        # nums[1] + nums[2] = 2 + 4 = 6
```

**Constraints:** `2 <= len(nums) <= 10^4`, exactly one valid answer exists.

### Approach 1 — Brute Force (worst complexity)

**Idea:** Try every pair `(i, j)` with `i < j` and check if `nums[i] + nums[j] == target`. This is the most direct translation of the problem into code, with no cleverness.

**Time complexity:** `O(n^2)` — for each of the `n` elements we scan up to `n` others.

**Space complexity:** `O(1)` — only a few loop variables; nothing scales with input size.

In [2]:
from typing import List

def two_sum_brute(nums: List[int], target: int) -> List[int]:
    n = len(nums)
    for i in range(n):                     # pick the first number of the pair
        for j in range(i + 1, n):          # pair it with every LATER number (j > i)
            if nums[i] + nums[j] == target:# do these two add up to the target?
                return [i, j]              # yes -> return their positions
    return []                              # (never reached: a solution is guaranteed)

### Approach 2 — Sort + Two Pointers (better)

**Idea:** Pair each value with its original index, sort by value, then move two pointers inward from both ends. If the current sum is too small, advance the left pointer (need a bigger value); if too big, retreat the right pointer. This trades the O(n²) scan for an O(n log n) sort plus an O(n) sweep.

**Time complexity:** `O(n log n)` — dominated by the sort; the two-pointer sweep is O(n).

**Space complexity:** `O(n)` — we store `(value, index)` pairs so we can recover the *original* indices after sorting.

In [3]:
from typing import List

def two_sum_better(nums: List[int], target: int) -> List[int]:
    # Pair each value with its ORIGINAL index, then sort by value.
    indexed = sorted((val, i) for i, val in enumerate(nums))
    lo, hi = 0, len(indexed) - 1           # two pointers: smallest and largest values
    while lo < hi:
        s = indexed[lo][0] + indexed[hi][0]# sum of the two ends
        if s == target:
            return sorted([indexed[lo][1], indexed[hi][1]])  # return original indices
        elif s < target:
            lo += 1                        # sum too small -> need a bigger left value
        else:
            hi -= 1                        # sum too big -> need a smaller right value
    return []

### Approach 3 — One-Pass Hash Map (optimal)

**Idea:** Walk the array once. For each `x`, compute the complement `target - x` and ask the hash map “have I seen this complement already?” If yes, we found the pair. If no, store `x` (with its index) and continue. Every lookup and insert is O(1) average.

**Time complexity:** `O(n)` — a single pass with O(1) hash operations per element.

**Space complexity:** `O(n)` — in the worst case the map holds nearly every element before the pair is found.

In [4]:
from typing import List

def two_sum_optimal(nums: List[int], target: int) -> List[int]:
    seen = {}                              # value -> index, for numbers we've already passed
    for i, x in enumerate(nums):
        need = target - x                  # the partner that would complete the pair
        if need in seen:                   # have we already passed that partner?
            return [seen[need], i]         # yes -> we found the pair
        seen[x] = i                        # no -> remember x so a future number can find it
    return []

In [5]:
# Verify all three approaches agree on a few cases.
tests = [
    ([2, 7, 11, 15], 9, [0, 1]),
    ([3, 2, 4], 6, [1, 2]),
    ([3, 3], 6, [0, 1]),
]

for nums, target, expected in tests:
    b = two_sum_brute(nums, target)
    m = two_sum_better(nums, target)
    o = two_sum_optimal(nums, target)
    print(f"nums={nums}, target={target} -> brute={b}, better={m}, optimal={o} | expected={expected}")
    assert b == expected and m == expected and sorted(o) == expected, "mismatch!"
print("\nAll tests passed")

nums=[2, 7, 11, 15], target=9 -> brute=[0, 1], better=[0, 1], optimal=[0, 1] | expected=[0, 1]
nums=[3, 2, 4], target=6 -> brute=[1, 2], better=[1, 2], optimal=[1, 2] | expected=[1, 2]
nums=[3, 3], target=6 -> brute=[0, 1], better=[0, 1], optimal=[0, 1] | expected=[0, 1]

All tests passed


## ⏱️ Empirically Checking the Complexities

You can't ask Python for a function's Big-O directly — but you **can measure it**. Time each solution on inputs of growing size `n` and watch the **doubling ratio**: how much the runtime grows each time `n` doubles.

| Theoretical | Expected time ratio when `n` → `2n` |
|-------------|-------------------------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |

We force the **worst case** by placing the answer pair at the very end, so no solution can exit early. The measured ratios below should track the table above (real numbers wobble due to CPU caching, constants, and machine noise).

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    # Answer is the LAST two elements, so no solution can short-circuit -> true worst case.
    nums = list(range(1, n + 1))
    target = nums[-1] + nums[-2]
    return (nums, target)

solutions = {
    "brute   O(n^2)":     two_sum_brute,
    "better  O(n log n)": two_sum_better,
    "optimal O(n)":       two_sum_optimal,
}
sizes = [1000, 2000, 4000, 8000]

# plot=True draws a log-log chart (needs matplotlib); set plot=False for table only.
benchmark(solutions, make_worst_case, sizes, plot=True)

## 🧩 Patterns Learned

- **Complement + Hash Map:** When a problem asks you to find two things that combine to a target, store what you've seen and look up the *complement* in O(1) instead of re-scanning. This turns O(n²) pair-search into O(n).
- **Signal to reach for it:** “find a pair/two elements that sum/match to X”, “does a value with property P exist”, or any repeated membership test inside a loop.
- **Two-pointer on sorted data:** A reusable fallback when O(1) extra space matters more than the O(n log n) sort cost, or when the array is already sorted.
- **Related problems:** 3Sum, Two Sum II (sorted input → pure two-pointer), Two Sum III (design), Subarray Sum Equals K (prefix-sum + hash map).
- **Common pitfalls:** (1) reusing the same index twice — check the complement *before* inserting the current element; (2) forgetting that sorting destroys original indices — carry the index alongside the value; (3) assuming values are unique when duplicates like `[3, 3]` are valid.